# LeetCode 1718 — Construct the Lexicographically Largest Valid Sequence bằng Python

Link đề: https://leetcode.com/problems/construct-the-lexicographically-largest-valid-sequence/ · Độ khó: **Medium** · Chủ đề: Array, Backtracking

Skeleton LeetCode:

```python
class Solution:
    def constructDistancedSequence(self, n: int) -> List[int]:
        ...
```

Lộ trình: **liệt kê mọi dãy hợp lệ rồi lấy max** (vét cạn, `n ≤ 8`) → **quay lui điền từng vị trí từ trái sang phải, thử số lớn trước** — nghiệm đầu tiên tìm được chính là nghiệm lớn nhất → tối ưu hằng số bằng **bitmask** → **tra bảng** (chỉ có 20 input khả dĩ).

## 1. Mục tiêu học

1. Hiểu cấu trúc dãy: số `1` xuất hiện **một lần**, mỗi số `i ∈ [2, n]` xuất hiện **hai lần** cách nhau đúng `i` ⇒ độ dài `2n − 1`.
2. Viết **quay lui (backtracking)** theo **vị trí**: tìm ô trống đầu tiên, thử đặt từng số, hoàn tác khi thất bại.
3. Hiểu vì sao "**duyệt vị trí từ trái, thử số từ lớn tới nhỏ, dừng ở nghiệm đầu tiên**" cho ra nghiệm **lớn nhất theo thứ tự từ điển**.
4. Cắt tỉa: kiểm tra ô `pos + i` còn trống và trong biên trước khi đặt; dùng **bitmask** cho tập số đã dùng.
5. Kiểm thử một bài "xây dựng" bằng **hàm kiểm tra hợp lệ** và so sánh với vét cạn trên `n` nhỏ.

## 2. Đề bài & ví dụ

Cho số nguyên `n`, tìm một dãy gồm các phần tử trong `[1, n]` thỏa mãn:
- Số `1` xuất hiện **đúng một lần**.
- Mỗi số nguyên từ `2` đến `n` xuất hiện **đúng hai lần**.
- Với mọi `i` trong `[2, n]`, **khoảng cách** giữa hai lần xuất hiện của `i` đúng bằng `i` (khoảng cách giữa `a[x]` và `a[y]` là `|y − x|`).

Trả về dãy **lớn nhất theo thứ tự từ điển**. Đề bảo đảm luôn có nghiệm.

Dãy `a` lớn hơn dãy `b` (cùng độ dài) nếu ở vị trí đầu tiên khác nhau, `a` có số lớn hơn. Ví dụ `[0,1,9,0] > [0,1,5,6]`.

**Ví dụ 1:** `n = 3` → `[3,1,2,3,2]`. (`[2,3,2,1,3]` cũng hợp lệ nhưng nhỏ hơn.)

**Ví dụ 2:** `n = 5` → `[5,3,1,4,3,5,2,4,2]`.

**Ràng buộc:** `1 ≤ n ≤ 20`.

In [1]:
import random
import time
from typing import List

examples = [
    (3, [3, 1, 2, 3, 2]),
    (5, [5, 3, 1, 4, 3, 5, 2, 4, 2]),
]
for n, want in examples:
    print(f"n={n} -> kỳ vọng {want}")

def is_valid(seq, n):
    """Kiểm tra seq có phải dãy hợp lệ cho n."""
    if len(seq) != 2 * n - 1:
        return False
    pos = {}
    for idx, v in enumerate(seq):
        if not (1 <= v <= n):
            return False
        pos.setdefault(v, []).append(idx)
    if pos.get(1, []) and len(pos[1]) != 1:
        return False
    for v in range(1, n + 1):
        p = pos.get(v, [])
        if v == 1:
            if len(p) != 1: return False
        elif len(p) != 2 or p[1] - p[0] != v:
            return False
    return True

print([is_valid(w, n) for n, w in examples], is_valid([2, 3, 2, 1, 3], 3), is_valid([3, 2, 1, 3, 2], 3))

n=3 -> kỳ vọng [3, 1, 2, 3, 2]
n=5 -> kỳ vọng [5, 3, 1, 4, 3, 5, 2, 4, 2]
[True, True] True False


## 3. Kiến thức nền

### 3.1 Độ dài dãy

`1` xuất hiện 1 lần, `2..n` mỗi số 2 lần ⇒ `1 + 2(n − 1) = 2n − 1` phần tử. Ta dùng mảng `seq = [0] * (2n − 1)`, `0` nghĩa là ô trống.

### 3.2 So sánh danh sách theo thứ tự từ điển

Python so sánh `list` đúng theo thứ tự từ điển: `max(list_of_lists)` cho dãy lớn nhất.

In [2]:
print([3, 1, 2, 3, 2] > [2, 3, 2, 1, 3], max([[2, 3, 2, 1, 3], [3, 1, 2, 3, 2]]))
print([0, 1, 9, 0] > [0, 1, 5, 6])

True [3, 1, 2, 3, 2]
True


### 3.3 Khuôn quay lui (backtracking)

```text
def dfs(trạng thái):
    if đã hoàn chỉnh: ghi nhận / return True
    for lựa chọn in các lựa chọn hợp lệ:
        áp dụng lựa chọn
        if dfs(trạng thái mới): return True     # dừng sớm nếu chỉ cần 1 nghiệm
        hoàn tác lựa chọn
    return False
```

In [3]:
# ví dụ nhỏ: liệt kê hoán vị của [1,2,3] bằng quay lui
def perms(arr):
    out, cur, used = [], [], [False] * len(arr)
    def dfs():
        if len(cur) == len(arr):
            out.append(cur[:]); return
        for i, x in enumerate(arr):
            if not used[i]:
                used[i] = True; cur.append(x)
                dfs()
                cur.pop(); used[i] = False
    dfs(); return out
print(perms([1, 2, 3]))

[[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### 3.4 Bitmask cho tập số đã dùng

`used >> v & 1` kiểm tra số `v` đã dùng; `used | (1 << v)` đánh dấu; `used & ~(1 << v)` (hoặc `^`) bỏ đánh dấu. Nhanh hơn `list[bool]` một chút và dễ truyền qua đệ quy.

In [4]:
used = 0
used |= 1 << 3; used |= 1 << 5
print(bin(used), used >> 3 & 1, used >> 4 & 1)
used ^= 1 << 3
print(bin(used))

0b101000 1 0
0b100000


## 4. Ý tưởng nền tảng / trực giác

Thử tay `n = 3` (độ dài 5). Muốn dãy **lớn nhất**, ô đầu tiên nên là số **lớn nhất có thể**: `3`. Đặt `3` ở ô 0 thì bản sao ở ô `0 + 3 = 3`:

`[3, _, _, 3, _]`

Ô trống đầu tiên là ô 1. Thử `2` ⇒ bản sao ở ô 3 — đã có `3` ⇒ không được. Thử `1` (chỉ một lần): `[3, 1, _, 3, _]`. Ô 2: thử `2` ⇒ bản sao ở ô 4 trống ⇒ `[3, 1, 2, 3, 2]`. Xong!

Đây chính là chiến lược quay lui: **ô trống trái nhất, số lớn nhất trước**. Nếu bị kẹt thì quay lui thử số nhỏ hơn ở ô gần nhất.

## 5. Cách 1 — Brute Force / baseline: liệt kê mọi dãy hợp lệ rồi lấy max

**Ý tưởng:** quay lui điền vị trí (thứ tự thử số tùy ý), **không dừng** ở nghiệm đầu, thu thập mọi nghiệm và trả về `max`. Đúng theo định nghĩa, không cần lập luận thứ tự.

In [5]:
def all_valid_sequences(n):
    L = 2 * n - 1
    seq = [0] * L
    used = [False] * (n + 1)
    out = []

    def dfs(pos):
        while pos < L and seq[pos]:
            pos += 1
        if pos == L:
            out.append(seq[:]); return
        for v in range(1, n + 1):
            if used[v]:
                continue
            if v == 1:
                seq[pos] = 1; used[1] = True
                dfs(pos + 1)
                seq[pos] = 0; used[1] = False
            elif pos + v < L and seq[pos + v] == 0:
                seq[pos] = seq[pos + v] = v; used[v] = True
                dfs(pos + 1)
                seq[pos] = seq[pos + v] = 0; used[v] = False

    dfs(0)
    return out

def construct_brute(n):
    return max(all_valid_sequences(n))

for n, want in examples:
    print(construct_brute(n), "kỳ vọng", want)

[3, 1, 2, 3, 2] kỳ vọng [3, 1, 2, 3, 2]
[5, 3, 1, 4, 3, 5, 2, 4, 2] kỳ vọng [5, 3, 1, 4, 3, 5, 2, 4, 2]


**Độ phức tạp:** số nghiệm và số nút tìm kiếm tăng rất nhanh theo `n` (mũ). Dùng được tới khoảng `n = 9`.

**Nút thắt:** ta chỉ cần **một** nghiệm — nghiệm lớn nhất. Nếu tìm kiếm theo đúng thứ tự từ điển giảm dần, nghiệm đầu tiên gặp được là đáp án ⇒ dừng sớm.

In [6]:
for n in range(1, 10):
    t0 = time.perf_counter(); sols = all_valid_sequences(n)
    print(f"n={n}: {len(sols):6d} dãy hợp lệ, {time.perf_counter() - t0:.3f}s")

n=1:      1 dãy hợp lệ, 0.000s
n=2:      1 dãy hợp lệ, 0.000s
n=3:      2 dãy hợp lệ, 0.000s
n=4:      6 dãy hợp lệ, 0.000s
n=5:     10 dãy hợp lệ, 0.000s
n=6:     22 dãy hợp lệ, 0.000s
n=7:     76 dãy hợp lệ, 0.001s
n=8:    364 dãy hợp lệ, 0.005s
n=9:   1876 dãy hợp lệ, 0.032s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `pos` là ô trống **trái nhất**. Mọi ô trước `pos` đã cố định. Nếu ta thử các giá trị cho ô `pos` theo thứ tự **giảm dần** và đệ quy theo cùng quy tắc, thì các nghiệm được liệt kê theo đúng **thứ tự từ điển giảm dần** ⇒ nghiệm đầu tiên là lớn nhất.

Một điểm tinh tế: các ô **sau** `pos` có thể đã được điền sẵn (bản sao của số đặt trước đó). Nhưng chúng được quyết định bởi các lựa chọn ở các ô trước `pos`, nên thứ tự từ điển vẫn được quyết định bởi giá trị tại `pos` (ô trống đầu tiên) — xem chứng minh ở mục 10.

**Quy tắc:**
- Nếu ô `pos` đã có số ⇒ nhảy sang ô kế.
- Thử `v = n, n−1, …, 2`: chỉ đặt được khi `v` chưa dùng, `pos + v < 2n − 1` và ô `pos + v` trống.
- Thử `v = 1` cuối cùng: chỉ cần chưa dùng.
- Nếu không số nào đi tiếp được ⇒ quay lui.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui tham lam theo vị trí, số lớn trước, dừng ở nghiệm đầu

In [7]:
def construct_greedy_bt(n):
    L = 2 * n - 1
    seq = [0] * L
    used = [False] * (n + 1)

    def dfs(pos):
        while pos < L and seq[pos]:
            pos += 1
        if pos == L:
            return True
        for v in range(n, 0, -1):
            if used[v]:
                continue
            if v == 1:
                seq[pos] = 1; used[1] = True
                if dfs(pos + 1): return True
                seq[pos] = 0; used[1] = False
            elif pos + v < L and seq[pos + v] == 0:
                seq[pos] = seq[pos + v] = v; used[v] = True
                if dfs(pos + 1): return True
                seq[pos] = seq[pos + v] = 0; used[v] = False
        return False

    dfs(0)
    return seq

for n, want in examples:
    print(construct_greedy_bt(n), "kỳ vọng", want)

[3, 1, 2, 3, 2] kỳ vọng [3, 1, 2, 3, 2]
[5, 3, 1, 4, 3, 5, 2, 4, 2] kỳ vọng [5, 3, 1, 4, 3, 5, 2, 4, 2]


Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** worst case vẫn mũ về lý thuyết, nhưng nhờ thứ tự thử "số lớn trước" và dừng sớm, thực tế chỉ thăm vài trăm tới vài nghìn nút với mọi `n ≤ 20` (mục 11 đo cụ thể). Space `O(n)` (mảng + độ sâu đệ quy ≤ `2n − 1`).

### 7.2 Cách 3 — Cùng thuật toán, dùng bitmask và vòng lặp nội tuyến

Lưu tập số đã dùng trong một số nguyên, truyền qua tham số; giảm truy cập list. Thêm bộ đếm số nút để đo.

In [8]:
def construct_bitmask(n, stats=None):
    L = 2 * n - 1
    seq = [0] * L
    nodes = [0]

    def dfs(pos, used):
        nodes[0] += 1
        while pos < L and seq[pos]:
            pos += 1
        if pos == L:
            return True
        for v in range(n, 1, -1):
            if not (used >> v & 1):
                q = pos + v
                if q < L and not seq[q]:
                    seq[pos] = seq[q] = v
                    if dfs(pos + 1, used | (1 << v)):
                        return True
                    seq[pos] = seq[q] = 0
        if not (used & 2):
            seq[pos] = 1
            if dfs(pos + 1, used | 2):
                return True
            seq[pos] = 0
        return False

    dfs(0, 0)
    if stats is not None:
        stats.append(nodes[0])
    return seq

for n, want in examples:
    print(construct_bitmask(n), "kỳ vọng", want)

[3, 1, 2, 3, 2] kỳ vọng [3, 1, 2, 3, 2]
[5, 3, 1, 4, 3, 5, 2, 4, 2] kỳ vọng [5, 3, 1, 4, 3, 5, 2, 4, 2]


**Độ phức tạp:** như Cách 2 về số nút; hằng số nhỏ hơn.

### 7.3 Cách 4 — Tra bảng (precompute)

Chỉ có 20 giá trị `n` khả dĩ. Có thể chạy Cách 3 **một lần** cho mọi `n` và lưu kết quả; khi nộp, trả về `TABLE[n]`. Về độ phức tạp là `O(1)` mỗi truy vấn (`O(n)` để sao chép kết quả). Đây là kỹ thuật hợp lệ khi miền input rất nhỏ, nhưng cần sinh bảng bằng một thuật toán đúng — ở đây ta sinh bảng ngay trong notebook.

In [9]:
TABLE = {n: construct_bitmask(n) for n in range(1, 21)}

def construct_table(n):
    return list(TABLE[n])

for n in [1, 2, 3, 4, 10, 20]:
    print(n, TABLE[n])

1 [1]
2 [2, 1, 2]
3 [3, 1, 2, 3, 2]
4 [4, 2, 3, 2, 4, 3, 1]
10 [10, 8, 6, 9, 3, 1, 7, 3, 6, 8, 10, 5, 9, 7, 4, 2, 5, 2, 4]
20 [20, 18, 19, 15, 13, 17, 10, 16, 7, 5, 3, 14, 12, 3, 5, 7, 10, 13, 15, 18, 20, 19, 17, 16, 12, 14, 11, 9, 4, 6, 8, 2, 4, 2, 1, 6, 9, 11, 8]


## 8. TRACE từng bước

Trace Cách 2 với `n = 4` (độ dài 7): in mỗi lần đặt / gỡ một số.

In [10]:
def trace_bt(n, limit=60):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1); lines = [0]

    def show(msg):
        if lines[0] < limit:
            print(msg); lines[0] += 1

    def dfs(pos, depth):
        while pos < L and seq[pos]:
            pos += 1
        if pos == L:
            show("  " * depth + f"HOÀN THÀNH: {seq}"); return True
        for v in range(n, 0, -1):
            if used[v]:
                continue
            q = pos if v == 1 else pos + v
            if v != 1 and (q >= L or seq[q]):
                show("  " * depth + f"ô {pos}: thử {v} -> ô {q} {'ngoài biên' if q >= L else 'bận'}, bỏ")
                continue
            seq[pos] = seq[q] = v; used[v] = True
            show("  " * depth + f"ô {pos}: đặt {v}" + (f" (bản sao ở ô {q})" if v != 1 else "") + f" -> {seq}")
            if dfs(pos + 1, depth + 1):
                return True
            seq[pos] = seq[q] = 0; used[v] = False
            show("  " * depth + f"ô {pos}: gỡ {v} (quay lui)")
        return False

    dfs(0, 0)
    return seq

trace_bt(4)

ô 0: đặt 4 (bản sao ở ô 4) -> [4, 0, 0, 0, 4, 0, 0]
  ô 1: thử 3 -> ô 4 bận, bỏ
  ô 1: đặt 2 (bản sao ở ô 3) -> [4, 2, 0, 2, 4, 0, 0]
    ô 2: đặt 3 (bản sao ở ô 5) -> [4, 2, 3, 2, 4, 3, 0]
      ô 6: đặt 1 -> [4, 2, 3, 2, 4, 3, 1]
        HOÀN THÀNH: [4, 2, 3, 2, 4, 3, 1]


[4, 2, 3, 2, 4, 3, 1]

Đọc trace: ô 0 đặt `4` (bản sao ở ô 4). Ô 1 thử `3` → bản sao ở ô 4 bận; thử `2` → ô 3; … Khi bị kẹt, thuật toán gỡ số ở ô gần nhất và thử số nhỏ hơn. Bảng tóm tắt các bước chính cho `n = 3`:

| bước | ô trống trái nhất | thử | kết quả | dãy |
|---|---|---|---|---|
| 1 | 0 | 3 | đặt, bản sao ô 3 | `[3,_,_,3,_]` |
| 2 | 1 | 2 | ô 3 bận ⇒ bỏ | |
| 3 | 1 | 1 | đặt | `[3,1,_,3,_]` |
| 4 | 2 | 2 | đặt, bản sao ô 4 | `[3,1,2,3,2]` ✓ |

In [11]:
trace_bt(3)

ô 0: đặt 3 (bản sao ở ô 3) -> [3, 0, 0, 3, 0]
  ô 1: thử 2 -> ô 3 bận, bỏ
  ô 1: đặt 1 -> [3, 1, 0, 3, 0]
    ô 2: đặt 2 (bản sao ở ô 4) -> [3, 1, 2, 3, 2]
      HOÀN THÀNH: [3, 1, 2, 3, 2]


[3, 1, 2, 3, 2]

## 9. Code nộp LeetCode

In [12]:
class Solution:
    def constructDistancedSequence(self, n: int) -> List[int]:
        L = 2 * n - 1
        seq = [0] * L
        used = [False] * (n + 1)

        def dfs(pos: int) -> bool:
            while pos < L and seq[pos]:
                pos += 1
            if pos == L:
                return True
            for v in range(n, 0, -1):
                if used[v]:
                    continue
                if v == 1:
                    seq[pos] = 1; used[1] = True
                    if dfs(pos + 1):
                        return True
                    seq[pos] = 0; used[1] = False
                elif pos + v < L and seq[pos + v] == 0:
                    seq[pos] = seq[pos + v] = v; used[v] = True
                    if dfs(pos + 1):
                        return True
                    seq[pos] = seq[pos + v] = 0; used[v] = False
            return False

        dfs(0)
        return seq

sol = Solution()
for n, want in examples:
    got = sol.constructDistancedSequence(n)
    print(got, "OK" if got == want else "SAI")

[3, 1, 2, 3, 2] OK
[5, 3, 1, 4, 3, 5, 2, 4, 2] OK


## 10. Vì sao thuật toán đúng

**Tính đầy đủ.** Mọi dãy hợp lệ `S` đều được thuật toán "đi qua" nếu không dừng sớm: xét ô trống trái nhất `pos` ở mỗi bước, giá trị `S[pos]` là một lựa chọn hợp lệ (nếu `S[pos] = v ≥ 2` là **lần xuất hiện đầu** của `v` — vì lần xuất hiện thứ hai ở `pos − v` thì ô `pos` đã bị điền sẵn rồi). Vậy quay lui liệt kê đúng tập mọi dãy hợp lệ (Cách 1 dựa trên điều này).

**Thứ tự liệt kê là từ điển giảm dần.** Xét hai dãy hợp lệ `A > B`, khác nhau lần đầu ở ô `p` với `A[p] > B[p]`. Trong cây tìm kiếm, cả hai đi chung đường cho tới khi lần đầu xét một ô trống `pos` mà hai dãy đặt số khác nhau. Mọi ô `< pos` đều đã cố định giống nhau; các ô được điền sẵn (bản sao) cũng giống nhau vì do cùng các lựa chọn trước đó. Vậy `pos` chính là `p` (ô đầu tiên khác nhau, và ô này đang trống trên cả hai nhánh). Thuật toán thử `A[p]` **trước** `B[p]` vì thử số giảm dần ⇒ nhánh chứa `A` được khám phá trước ⇒ mọi nghiệm lớn hơn được gặp trước.

⇒ Nghiệm **đầu tiên** tìm được là nghiệm **lớn nhất**. Đề bảo đảm có nghiệm nên `dfs(0)` luôn trả về `True`.

Mục 12 kiểm chứng bằng cách so sánh với `max(all_valid_sequences(n))` cho `n ≤ 9`.

## 11. Phân tích Time / Space

Gọi `n ≤ 20`, `L = 2n − 1`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê mọi nghiệm | mũ (≈ số nút toàn cây) | `O(L · #nghiệm)` | `n ≤ 9` |
| 2. Quay lui tham lam | mũ worst, rất nhỏ thực tế | `O(L)` | **nộp bài** |
| 3. Bitmask | như 2, hằng số nhỏ hơn | `O(L)` | |
| 4. Tra bảng | `O(L)` mỗi truy vấn | `O(n²)` bảng | chỉ khi miền input nhỏ |

Số nút thực tế của Cách 3:

In [13]:
total_t = 0
for n in range(1, 21):
    st = []
    t0 = time.perf_counter(); construct_bitmask(n, st); dt = time.perf_counter() - t0
    total_t += dt
    print(f"n={n:2d}: {st[0]:6d} nút, {dt * 1000:7.2f} ms")
print(f"tổng thời gian cho mọi n: {total_t:.3f}s")

n= 1:      2 nút,    0.00 ms
n= 2:      3 nút,    0.00 ms
n= 3:      4 nút,    0.00 ms
n= 4:      5 nút,    0.00 ms
n= 5:      8 nút,    0.00 ms
n= 6:      9 nút,    0.00 ms
n= 7:     13 nút,    0.01 ms
n= 8:     31 nút,    0.01 ms
n= 9:     74 nút,    0.03 ms
n=10:    315 nút,    0.15 ms
n=11:     62 nút,    0.03 ms
n=12:     63 nút,    0.03 ms
n=13:    154 nút,    0.11 ms
n=14:    107 nút,    0.06 ms
n=15:     70 nút,    0.04 ms
n=16:    224 nút,    0.14 ms
n=17:   1214 nút,    0.81 ms
n=18:   1203 nút,    0.87 ms
n=19:   2603 nút,    1.79 ms
n=20:    766 nút,    0.57 ms
tổng thời gian cho mọi n: 0.005s


## 12. Bộ test

In [14]:
fns = [construct_greedy_bt, construct_bitmask, construct_table, lambda n: Solution().constructDistancedSequence(n)]

for n, want in examples:
    for fn in fns:
        assert fn(n) == want

# edge case: n = 1, 2
assert Solution().constructDistancedSequence(1) == [1]
assert Solution().constructDistancedSequence(2) == [2, 1, 2]

# so với vét cạn (max mọi nghiệm) cho n <= 9
for n in range(1, 10):
    want = construct_brute(n)
    for fn in fns:
        assert fn(n) == want, n
print("n = 1..9 khớp max của mọi dãy hợp lệ")

# n = 10..20: kiểm tra tính hợp lệ và nhất quán giữa các cách
for n in range(1, 21):
    res = [fn(n) for fn in fns]
    assert all(r == res[0] for r in res) and is_valid(res[0], n), n
    assert res[0][0] == n                           # dãy lớn nhất luôn bắt đầu bằng n
print("n = 1..20: mọi cách cho cùng một dãy hợp lệ, bắt đầu bằng n")

n = 1..9 khớp max của mọi dãy hợp lệ
n = 1..20: mọi cách cho cùng một dãy hợp lệ, bắt đầu bằng n


In [15]:
# kiểm tra "cục bộ tối ưu" cho n lớn: không thể tăng bất kỳ tiền tố nào
# (với mỗi vị trí p, thử ép tiền tố res[:p] + [x] với x > res[p], xem có nghiệm không)
def exists_with_prefix(n, prefix):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    # đặt tiền tố
    for p, v in enumerate(prefix):
        if seq[p]:
            if seq[p] != v: return False
            continue
        if used[v]: return False
        q = p if v == 1 else p + v
        if q >= L or seq[q]: return False
        seq[p] = seq[q] = v; used[v] = True
    def dfs(pos):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(n, 0, -1):
            if used[v]: continue
            q = pos if v == 1 else pos + v
            if q < L and (v == 1 or not seq[q]):
                seq[pos] = seq[q] = v; used[v] = True
                if dfs(pos + 1): return True
                seq[pos] = seq[q] = 0; used[v] = False
        return False
    return dfs(0)

for n in [10, 12]:
    res = construct_bitmask(n)
    for p in range(len(res)):
        for x in range(res[p] + 1, n + 1):
            assert not exists_with_prefix(n, res[:p] + [x]), (n, p, x)
print("n = 10, 12: không tồn tại dãy hợp lệ nào lớn hơn (kiểm tra từng tiền tố)")

n = 10, 12: không tồn tại dãy hợp lệ nào lớn hơn (kiểm tra từng tiền tố)


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Duyệt theo **số** (đặt số `n` trước, rồi `n−1`, …) thay vì theo **vị trí**

Đặt mỗi số vào vị trí trống trái nhất có thể, theo thứ tự số giảm dần. Đúng tính hợp lệ (nếu có quay lui) nhưng nghiệm đầu tiên **không** chắc là lớn nhất theo từ điển.

In [16]:
def wrong_by_number(n):
    L = 2 * n - 1; seq = [0] * L
    def dfs(v):
        if v == 0:
            return True
        for p in range(L):                       # thử vị trí trái nhất trước
            q = p if v == 1 else p + v
            if q < L and seq[p] == 0 and seq[q] == 0:
                seq[p] = seq[q] = v
                if dfs(v - 1): return True
                seq[p] = seq[q] = 0
        return False
    dfs(n)
    return seq

for n in range(3, 9):
    w, r = wrong_by_number(n), construct_greedy_bt(n)
    if w != r:
        print(f"n={n}: sai = {w} (hợp lệ: {is_valid(w, n)})\n      đúng = {r}")
        break

n=5: sai = [5, 2, 4, 2, 3, 5, 4, 3, 1] (hợp lệ: True)
      đúng = [5, 3, 1, 4, 3, 5, 2, 4, 2]


**Sửa:** quay lui theo **ô trống trái nhất**, thử số giảm dần — để mỗi quyết định ứng với một vị trí trong thứ tự từ điển.

### Lỗi 2 — Thử số `1` như các số khác (đặt bản sao ở `pos + 1`)

Số `1` chỉ xuất hiện một lần. Nếu xử lý `1` như các số khác (chiếm cả ô `pos + 1`), tổng số ô cần dùng là `2n` trong khi mảng chỉ có `2n − 1` ô ⇒ không bao giờ lấp đầy, quay lui duyệt hết cây rồi báo thất bại.

In [17]:
def wrong_one_twice(n):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    def dfs(pos):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(n, 0, -1):
            if not used[v] and pos + v < L and seq[pos + v] == 0:   # SAI: số 1 cũng cần ô pos+1
                seq[pos] = seq[pos + v] = v; used[v] = True
                if dfs(pos + 1): return True
                seq[pos] = seq[pos + v] = 0; used[v] = False
        return False
    ok = dfs(0)
    return seq, ok

for n in [3, 4, 5]:
    seq, ok = wrong_one_twice(n)
    print(f"n={n}: tìm được? {ok}, dãy = {seq}, hợp lệ? {is_valid(seq, n)} | đúng = {construct_greedy_bt(n)}")

n=3: tìm được? False, dãy = [0, 0, 0, 0, 0], hợp lệ? False | đúng = [3, 1, 2, 3, 2]
n=4: tìm được? False, dãy = [0, 0, 0, 0, 0, 0, 0], hợp lệ? False | đúng = [4, 2, 3, 2, 4, 3, 1]
n=5: tìm được? False, dãy = [0, 0, 0, 0, 0, 0, 0, 0, 0], hợp lệ? False | đúng = [5, 3, 1, 4, 3, 5, 2, 4, 2]


**Sửa:** xử lý riêng `v == 1`: chỉ cần ô `pos`.

### Lỗi 3 — Quên hoàn tác khi quay lui

Không gỡ `seq[pos]`, `seq[pos + v]` và `used[v]` khi nhánh thất bại ⇒ các ô bị "kẹt" giá trị cũ.

In [18]:
def wrong_no_undo(n):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    def dfs(pos):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(n, 0, -1):
            if used[v]: continue
            q = pos if v == 1 else pos + v
            if q < L and (v == 1 or seq[q] == 0):
                seq[pos] = seq[q] = v; used[v] = True
                if dfs(pos + 1): return True
                # SAI: thiếu seq[pos] = seq[q] = 0; used[v] = False
        return False
    ok = dfs(0)
    return seq, ok

for n in [4, 5, 6]:
    seq, ok = wrong_no_undo(n)
    print(f"n={n}: trả về {ok}, dãy {seq}, hợp lệ? {is_valid(seq, n)}")

n=4: trả về True, dãy [4, 2, 3, 2, 4, 3, 1], hợp lệ? True
n=5: trả về False, dãy [5, 3, 4, 1, 3, 5, 4, 0, 0], hợp lệ? False
n=6: trả về False, dãy [6, 4, 2, 1, 2, 4, 6, 5, 0, 0, 0], hợp lệ? False


**Sửa:** luôn hoàn tác đầy đủ sau khi đệ quy trả về `False`.

### Lỗi 4 — Không bỏ qua ô đã điền sẵn

Bắt đầu thử số ở ô `pos` dù ô đó đã chứa bản sao của số trước ⇒ ghi đè.

In [19]:
def wrong_no_skip(n):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    def dfs(pos):
        if pos == L: return True                        # SAI: thiếu while seq[pos]
        for v in range(n, 0, -1):
            if used[v]: continue
            q = pos if v == 1 else pos + v
            if q < L and (v == 1 or seq[q] == 0):
                old = seq[pos]
                seq[pos] = seq[q] = v; used[v] = True
                if dfs(pos + 1): return True
                seq[pos] = old; seq[q] = 0 if q != pos else old; used[v] = False
        return False
    ok = dfs(0)
    return seq, ok

seq, ok = wrong_no_skip(3)
print("n=3:", seq, "tìm được?", ok, "hợp lệ?", is_valid(seq, 3))

n=3: [0, 0, 0, 0, 0] tìm được? False hợp lệ? False


**Sửa:** `while pos < L and seq[pos]: pos += 1` ở đầu `dfs`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tìm nghiệm **lớn/nhỏ nhất theo thứ tự từ điển** trong không gian tổ hợp" ⇒ quay lui **theo vị trí**, thử giá trị theo **thứ tự mong muốn**, **dừng ở nghiệm đầu tiên**.
- `n ≤ 20` + "đảm bảo có nghiệm" ⇒ quay lui là đủ; đừng tìm công thức.
- Kiểm tra ràng buộc **sớm nhất có thể** (ô bản sao trống & trong biên) để cắt tỉa.
- Miền input rất nhỏ ⇒ có thể **tra bảng** (sinh bằng chương trình đúng).
- Họ bài: **Langford pairing** / **Skolem sequence** (dãy có hai bản sao cách nhau `k`), Sudoku, N-Queens.

## 15. Biến thể của bài

1. **Dãy nhỏ nhất theo từ điển:** thử số **tăng dần** — cẩn thận: không phải mọi ô đầu đều có thể là `1`; quay lui vẫn đúng.
2. **Đếm số dãy hợp lệ** cho mỗi `n`: bỏ dừng sớm, đếm lá (Cách 1).
3. **Langford pairing `L(2, n)`:** mỗi số `1..n` xuất hiện hai lần, giữa hai số `k` có đúng `k` số khác (khoảng cách `k + 1`). Có nghiệm khi và chỉ khi `n ≡ 0` hoặc `3 (mod 4)`.
4. **Skolem sequence:** mỗi `k ∈ [1, n]` xuất hiện hai lần cách nhau `k`, độ dài `2n`; tồn tại khi `n ≡ 0, 1 (mod 4)`.

In [20]:
def count_valid(n):
    return len(all_valid_sequences(n))

def langford(n):
    """Biến thể 3: trả về một Langford pairing hoặc None."""
    L = 2 * n; seq = [0] * L
    def dfs(pos, used):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(n, 0, -1):
            q = pos + v + 1
            if not (used >> v & 1) and q < L and not seq[q]:
                seq[pos] = seq[q] = v
                if dfs(pos + 1, used | (1 << v)): return True
                seq[pos] = seq[q] = 0
        return False
    return seq if dfs(0, 0) else None

print("số dãy hợp lệ của bài gốc, n=1..8:", [count_valid(n) for n in range(1, 9)])
print("Langford tồn tại với n=1..8:", [langford(n) is not None for n in range(1, 9)])
print("Langford n=3:", langford(3), "| n=4:", langford(4))

số dãy hợp lệ của bài gốc, n=1..8: [1, 1, 2, 6, 10, 22, 76, 364]
Langford tồn tại với n=1..8: [False, False, True, True, False, False, True, True]
Langford n=3: [3, 1, 2, 1, 3, 2] | n=4: [4, 1, 3, 1, 2, 4, 3, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê + max | quay lui đầy đủ | mũ | lớn | kiểm chứng `n ≤ 9` |
| 2. Quay lui tham lam | ô trái nhất, số lớn trước, dừng sớm | rất nhỏ thực tế | `O(n)` | **nộp bài** |
| 3. Bitmask | như 2 | như 2 | `O(n)` | tối ưu hằng số |
| 4. Tra bảng | sinh sẵn 20 đáp án | `O(n)` | `O(n²)` | thi đấu, miền input nhỏ |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `constructDistancedSequence(2)` và `constructDistancedSequence(4)`?

**Bài 2 (tính tay):** lần theo quay lui cho `n = 4`: ô 1 thử những số nào trước khi đặt được?

**Bài 3 (điền TODO):** hoàn thiện `smallest_sequence(n)` trả về dãy hợp lệ **nhỏ nhất** theo từ điển.

**Bài 4 (tự viết lại):** viết lại Cách 2 **không dùng đệ quy** (dùng stack tường minh lưu `(pos, v_tiếp_theo_cần_thử)`).

**Bài 5 (mini challenge):** viết hàm `skolem(n)` tìm một Skolem sequence độ dài `2n` (mỗi `k ∈ [1, n]` xuất hiện hai lần cách nhau `k`) hoặc `None`; in kết quả với `n = 1..9` và kiểm tra điều kiện tồn tại `n mod 4 ∈ {0, 1}`.

In [21]:
def smallest_sequence(n):
    # TODO: như Cách 2 nhưng thử v = 1, 2, ..., n
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `n = 2`: độ dài 3. (2) Đặt 2 ở ô 0 và 2, ô 1 là 1. (3) `[2,1,2]`; `n = 4`: chạy Cách 2 (kết quả `[4,2,3,2,4,3,1]`).

**Bài 2:** (1) Ô 0 = 4 (bản sao ô 4). (2) Ô 1 thử 3 → ô 4 bận; thử 2 → ô 3 trống ⇒ đặt. (3) Xem trace mục 8.

**Bài 3:** (1) Đổi `range(n, 0, -1)` thành `range(1, n + 1)`. (2) Số 1 vẫn xử lý riêng. (3) So sánh với `min(all_valid_sequences(n))`.

**Bài 4:** (1) Stack chứa các quyết định đã làm `(pos, v)`. (2) Khi quay lui, pop quyết định, gỡ, thử `v − 1`. (3) Cẩn thận nhảy qua ô đã điền.

**Bài 5:** (1) Khoảng cách `k` cho cả `k = 1`: bản sao ở `pos + 1`. (2) Độ dài `2n`, quay lui như bài gốc. (3) So `n % 4`.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert Solution().constructDistancedSequence(2) == [2, 1, 2]
assert Solution().constructDistancedSequence(4) == [4, 2, 3, 2, 4, 3, 1]

# Bài 3
def smallest_sequence_ans(n):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    def dfs(pos):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(1, n + 1):
            if used[v]: continue
            q = pos if v == 1 else pos + v
            if q < L and (v == 1 or not seq[q]):
                seq[pos] = seq[q] = v; used[v] = True
                if dfs(pos + 1): return True
                seq[pos] = seq[q] = 0; used[v] = False
        return False
    dfs(0); return seq

for n in range(1, 9):
    assert smallest_sequence_ans(n) == min(all_valid_sequences(n))

# Bài 4 — không đệ quy
def construct_iterative(n):
    L = 2 * n - 1; seq = [0] * L; used = [False] * (n + 1)
    stack = []                      # (pos, v) đã đặt
    pos, start_v = 0, n
    while True:
        while pos < L and seq[pos]:
            pos += 1
        if pos == L:
            return seq
        placed = False
        for v in range(start_v, 0, -1):
            if used[v]: continue
            q = pos if v == 1 else pos + v
            if q < L and (v == 1 or not seq[q]):
                seq[pos] = seq[q] = v; used[v] = True
                stack.append((pos, v)); pos, start_v = pos + 1, n
                placed = True
                break
        if not placed:
            p, v = stack.pop()                     # quay lui
            q = p if v == 1 else p + v
            seq[p] = seq[q] = 0; used[v] = False
            pos, start_v = p, v - 1

for n in range(1, 21):
    assert construct_iterative(n) == TABLE[n]
print("Bài 1, 3, 4: OK")

Bài 1, 3, 4: OK


In [23]:
# Bài 5 — Skolem sequence
def skolem(n):
    L = 2 * n; seq = [0] * L
    def dfs(pos, used):
        while pos < L and seq[pos]: pos += 1
        if pos == L: return True
        for v in range(n, 0, -1):
            q = pos + v
            if not (used >> v & 1) and q < L and not seq[q]:
                seq[pos] = seq[q] = v
                if dfs(pos + 1, used | (1 << v)): return True
                seq[pos] = seq[q] = 0
        return False
    return seq if dfs(0, 0) else None

for n in range(1, 10):
    s = skolem(n)
    assert (s is not None) == (n % 4 in (0, 1)), n
    print(n, s)
print("Bài 5: điều kiện tồn tại n mod 4 ∈ {0,1} khớp với n = 1..9")

1 [1, 1]
2 None
3 None
4 [4, 2, 3, 2, 4, 3, 1, 1]
5 [5, 2, 4, 2, 3, 5, 4, 3, 1, 1]
6 None
7 None
8 [8, 6, 4, 7, 1, 1, 4, 6, 8, 5, 7, 2, 3, 2, 5, 3]
9 [9, 7, 8, 4, 2, 6, 2, 4, 7, 9, 8, 6, 5, 3, 1, 1, 3, 5]
Bài 5: điều kiện tồn tại n mod 4 ∈ {0,1} khớp với n = 1..9


## 20. Template nhớ nhanh + Checklist

```python
# Quay lui cho nghiệm lớn nhất theo từ điển
def dfs(pos):
    while pos < L and seq[pos]: pos += 1          # bỏ qua ô đã điền
    if pos == L: return True                      # nghiệm đầu tiên = lớn nhất
    for v in range(n, 0, -1):                     # thử giá trị lớn trước
        if used[v]: continue
        q = pos if v == 1 else pos + v
        if q < L and (v == 1 or seq[q] == 0):
            seq[pos] = seq[q] = v; used[v] = True
            if dfs(pos + 1): return True
            seq[pos] = seq[q] = 0; used[v] = False   # hoàn tác
    return False
```

**Checklist trước khi submit:**
- [ ] Độ dài mảng `2n − 1`.
- [ ] Duyệt theo **ô trống trái nhất**, thử số **giảm dần**, dừng ở nghiệm đầu.
- [ ] Số `1` chỉ chiếm **một** ô.
- [ ] Kiểm tra `pos + v < L` và ô `pos + v` trống trước khi đặt.
- [ ] Hoàn tác đủ `seq[pos]`, `seq[pos + v]`, `used[v]`.
- [ ] Thử `n = 1` (`[1]`) và `n = 2` (`[2,1,2]`).